In [ ]:
# import libraries
import numpy as np
import pandas as pd
from pathlib import Path

In [ ]:
# Select Platform
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

In [ ]:
#Project Path
if IN_COLAB:
    PROJECT_ROOT = Path("/content/drive/MyDrive/Final_Thesis")

else:
    PROJECT_ROOT = Path.cwd()
print("Running in Colab:", IN_COLAB)
print("Project root:", PROJECT_ROOT)

In [ ]:
drone_traj = PROJECT_ROOT/"drone_trajectory_dataset.csv"
gnss_traj = PROJECT_ROOT/"station_gnss_data.csv"

In [ ]:
drone_df = pd.read_csv(drone_traj)
gnss_df = pd.read_csv(gnss_traj)

In [ ]:
drone_df

In [ ]:

# 1. Sort data

drone_df = drone_df.sort_values(["track_id", "frame_idx"]).reset_index(drop=True)


# 2. Bounding box size


drone_df["w"] = drone_df["x2"] - drone_df["x1"]
drone_df["h"] = drone_df["y2"] - drone_df["y1"]

# Approximate bus length in pixels
# This works for horizontal, vertical, and inclined buses
drone_df["bbox_length"] = np.maximum(drone_df["w"], drone_df["h"])


# 3. Estimate bus direction using 0.5-second lookback


fps = 29.97  # the video has fps 29.97
lookback_seconds = 1 #change from 0.5 to 1
lookback_frames = int(fps * lookback_seconds)

# Position of the same bus 0.5 seconds ago
drone_df["x_prev"] = drone_df.groupby("track_id")["x_center"].shift(lookback_frames)
drone_df["y_prev"] = drone_df.groupby("track_id")["y_center"].shift(lookback_frames)

# Direction vector from previous position to current position
drone_df["dx"] = drone_df["x_center"] - drone_df["x_prev"]
drone_df["dy"] = drone_df["y_center"] - drone_df["y_prev"]

# Direction length
drone_df["dir_len"] = np.sqrt(drone_df["dx"]**2 + drone_df["dy"]**2)



# 4. Normalize direction vector

# Avoid division by zero
drone_df["ux"] = np.where(
    drone_df["dir_len"] > 0,
    drone_df["dx"] / drone_df["dir_len"],
    np.nan
)

drone_df["uy"] = np.where(
    drone_df["dir_len"] > 0,
    drone_df["dy"] / drone_df["dir_len"],
    np.nan
)

# Fill missing direction values
# This handles the first few frames and stopping cases
drone_df[["ux", "uy"]] = (
    drone_df.groupby("track_id")[["ux", "uy"]]
    .ffill()
    .bfill()
)


# 5. Shift bbox center toward estimated bus front

# k controls how far to shift from bbox center
# k = 0.00 -> bbox center
# k = 0.05 -> small front shift
# k = 0.10 -> moderate front shift
# k = 0.20 -> strong front shift
k = 0

drone_df["lambda_px"] = k * drone_df["bbox_length"]

# IMPORTANT:
# Use minus sign because your previous plot showed this direction
# moved the adjusted points closer to GNSS.
drone_df["x_front"] = drone_df["x_center"] - drone_df["lambda_px"] * drone_df["ux"]
drone_df["y_front"] = drone_df["y_center"] - drone_df["lambda_px"] * drone_df["uy"]



# 6. Save output

drone_df.to_csv("tracking_with_lookback_front_point.csv", index=False)

drone_df[[
    "frame_idx", "track_id",
    "x_center", "y_center",
    "x_front", "y_front",
    "bbox_length", "lambda_px",
    "ux", "uy", "dir_len"
]].head()

,frame_idx,track_id,x_center,y_center,x_front,y_front,bbox_length,lambda_px,ux,uy,dir_len
0,1,1,742.253418,448.556946,742.253418,448.556946,172.215759,0.0,-0.999858,-0.01688,NaN
1,2,1,742.260376,448.542847,742.260376,448.542847,172.218201,0.0,-0.999858,-0.01688,NaN
2,3,1,742.262939,448.528107,742.262939,448.528107,172.221619,0.0,-0.999858,-0.01688,NaN
3,4,1,742.273315,448.531128,742.273315,448.531128,172.177063,0.0,-0.999858,-0.01688,NaN
4,5,1,742.282593,448.559448,742.282593,448.559448,172.154358,0.0,-0.999858,-0.01688,NaN


In [ ]:
#copy the dataset
df = drone_df.copy()

# remove microseconds from time column
df['time'] = pd.to_datetime(df['real_time_str']).dt.strftime('%H:%M:%S')

# drop some columns
#df = df.drop(columns=['w', 'h', 'bbox_length', 'dx', 'dy', 'move_len', 'ux', 'uy', 'lambda_px','real_time','x1','x2','y1','y2','real_time_str'])
df.head()

/tmp/ipykernel_4280/65673710.py:5: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['time'] = pd.to_datetime(df['real_time_str']).dt.strftime('%H:%M:%S')


,frame_idx,time_sec,track_id,x1,y1,x2,y2,x_center,y_center,real_time,...,y_prev,dx,dy,dir_len,ux,uy,lambda_px,x_front,y_front,time
0,1,0.000000,1,656.145508,432.507385,828.361267,464.606506,742.253418,448.556946,2026-03-12 06:58:00.894000,...,NaN,NaN,NaN,NaN,-0.999858,-0.01688,0.0,742.253418,448.556946,06:58:00
1,2,0.033367,1,656.151245,432.469177,828.369446,464.616486,742.260376,448.542847,2026-03-12 06:58:00.927367,...,NaN,NaN,NaN,NaN,-0.999858,-0.01688,0.0,742.260376,448.542847,06:58:00
2,3,0.066733,1,656.152100,432.447540,828.373718,464.608673,742.262939,448.528107,2026-03-12 06:58:00.960733,...,NaN,NaN,NaN,NaN,-0.999858,-0.01688,0.0,742.262939,448.528107,06:58:00
3,4,0.100100,1,656.184814,432.410614,828.361877,464.651642,742.273315,448.531128,2026-03-12 06:58:00.994100,...,NaN,NaN,NaN,NaN,-0.999858,-0.01688,0.0,742.273315,448.531128,06:58:00
4,5,0.133467,1,656.205444,432.459961,828.359802,464.658905,742.282593,448.559448,2026-03-12 06:58:01.027467,...,NaN,NaN,NaN,NaN,-0.999858,-0.01688,0.0,742.282593,448.559448,06:58:01


In [ ]:
import cv2
import numpy as np
import os


# ---------- Ground control points ----------
# Image points: pixel coordinates [x, y]
image_points = np.array([
    [376.96, 901.18],
    [604.11, 352.48],
    [1375.22, 316.35],
    [1297.00, 811.88]
], dtype=np.float32)

world_points = np.array([
    [59.568888, 17.532215],
    [59.568749, 17.531307],
    [59.569210, 17.530506],
    [59.569411, 17.531193]  #the best coordination, don't lose
], dtype=np.float32)

# ---------- Compute homography ----------
H, status = cv2.findHomography(image_points, world_points)


# ---------- Extract pixel center coordinates ----------
pixels = df[['x_front', 'y_front']].values.astype(np.float32)
pixels = pixels.reshape(-1, 1, 2)

# ---------- Transform pixel coordinates to GPS ----------
latlon = cv2.perspectiveTransform(pixels, H)

# Since world_points are in [latitude, longitude] format
df['latitude'] = latlon[:, 0, 0]
df['longitude'] = latlon[:, 0, 1]

# ---------- Save output CSV ----------
output_dir = "/content/drive/MyDrive/Final_Thesis/runs/PositionValue"
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, "try_center_dronedata.csv")
df.to_csv(output_path, index=False)

print("Homography conversion complete.")
print(f"Output saved to: {output_path}")

Homography conversion complete.
Output saved to: /content/drive/MyDrive/Final_Thesis/runs/PositionValue/try_center_dronedata.csv


In [ ]:
traj = pd.read_csv("/content/drive/MyDrive/Final_Thesis/runs/PositionValue/try_center_dronedata.csv")
traj.head()

In [ ]:
#aggregate the data
# STEP 1: convert time to integer seconds
traj["sec"] = traj["time_sec"].astype(int)

# STEP 2: group by vehicle (track_id) AND second
traj_1hz = traj.groupby(["track_id", "sec"]).agg({
    "latitude": "mean",
    "longitude": "mean",
    "frame_idx": "first",
    "time": "first"
}).reset_index()

print(traj_1hz.head())

   track_id  sec   latitude  longitude  frame_idx      time
0         1    0  59.568886  17.531290          1  06:58:00
1         1    1  59.568883  17.531292         31  06:58:01
2         1    2  59.568883  17.531293         61  06:58:02
3         1    3  59.568886  17.531295         91  06:58:03
4         1    4  59.568886  17.531295        121  06:58:04


In [ ]:
traj_1hz.to_csv("/content/drive/MyDrive/Final_Thesis/runs/PositionValue/try_center_dronedata_1hz.csv", index=False)

# GNSS

In [ ]:
gnss_df

In [ ]:
from datetime import datetime
gnss_df['time'] = pd.to_datetime(gnss_df['datetime_local'])
gnss_df["time"] = gnss_df["time"].dt.strftime("%H:%M:%S")
gnss_df.head()

,operator,timestamp,vehicle_id,trip_id,route_id,lat,lon,speed_mps,bearing,datetime_utc,datetime_local,distance_to_balsta_m,time
0,ul,1.773295e+09,9031003001908240,NaN,NaN,59.568874,17.531410,0.0,139,2026/03/12 05:59:29+00,2026/03/12 06:59:29+01,77.318756,06:59:29
1,ul,1.773295e+09,9031003001908240,NaN,NaN,59.568874,17.531412,0.0,139,2026/03/12 05:59:36+00,2026/03/12 06:59:36+01,77.210984,06:59:36
2,ul,1.773295e+09,9031003001908240,NaN,NaN,59.568878,17.531412,0.0,139,2026/03/12 05:59:16+00,2026/03/12 06:59:16+01,77.203174,06:59:16
3,ul,1.773295e+09,9031003001908240,NaN,NaN,59.568874,17.531412,0.0,139,2026/03/12 05:59:22+00,2026/03/12 06:59:22+01,77.210984,06:59:22
4,ul,1.773295e+09,9031003001908240,NaN,NaN,59.568878,17.531408,0.0,139,2026/03/12 05:58:52+00,2026/03/12 06:58:52+01,77.418739,06:58:52


In [ ]:
drone_1hz = pd.read_csv("/content/drive/MyDrive/Final_Thesis/runs/PositionValue/try_center_dronedata_1hz.csv")
drone_1hz.head()

In [ ]:
merged_df = pd.merge(drone_1hz, gnss_df, on=['time'])
merged_df.head()

In [ ]:
filtered = merged_df[merged_df['track_id'] == 1]
filtered.head(100)